In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import layers, models
from keras import regularizers
from keras.applications import MobileNetV2,ResNet50V2,InceptionV3
from keras.applications.resnet50 import preprocess_input

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sanknn/face-mask-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'face-mask-detection' dataset.
Path to dataset files: /kaggle/input/face-mask-detection


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_dir = "/kaggle/input/face-mask-detection/Facemaskdetection/Facemaskdetection/train"
val_dir = "/kaggle/input/face-mask-detection/Facemaskdetection/Facemaskdetection/val"
test_dir = "/kaggle/input/face-mask-detection/Facemaskdetection/Facemaskdetection/test"
datagen = ImageDataGenerator(rescale=1./255)

train_data = datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='rgb'
)

val_data = datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='rgb'
)

test_data = datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    color_mode='rgb'
)

Found 8118 images belonging to 2 classes.
Found 1822 images belonging to 2 classes.
Found 1809 images belonging to 2 classes.


In [ ]:
# Model 1: Transfer Learning
base_model = ResNet50V2(input_shape=(IMG_SIZE,IMG_SIZE,3),
                         include_top=False,
                         weights='imagenet')

base_model.trainable = True
for layer in base_model.layers[:40]:
    layer.trainable = False
model1=models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1,activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])




94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [ ]:
# callback
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,restore_best_weights=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint('models/model1.h5', save_best_only=True)
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)

In [ ]:
model1.fit(train_data, validation_data=val_data, epochs=2,callbacks=[early_stopping,checkpoint,lr_scheduler])


Epoch 1/2
 12/254 ━━━━━━━━━━━━━━━━━━━━ 1:27 363ms/step - accuracy: 0.8398 - loss: 0.5983

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


254/254 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - accuracy: 0.9435 - loss: 0.3248

254/254 ━━━━━━━━━━━━━━━━━━━━ 181s 526ms/step - accuracy: 0.9617 - loss: 0.2209 - val_accuracy: 0.9561 - val_loss: 0.2120 - learning_rate: 0.0010
Epoch 2/2
254/254 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step - accuracy: 0.9781 - loss: 0.1084

254/254 ━━━━━━━━━━━━━━━━━━━━ 67s 261ms/step - accuracy: 0.9787 - loss: 0.0993 - val_accuracy: 0.9918 - val_loss: 0.0409 - learning_rate: 0.0010


In [ ]:
!pip install gradio
import gradio as gr

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):

    js = Javascript('''
        async function takePhoto(quality) {
            const video = document.createElement('video');
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(video);
            video.srcObject = stream;
            await video.play();

            await new Promise(resolve => setTimeout(resolve, 1000));

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;

            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getTracks().forEach(track => track.stop());
            video.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)
    data = eval_js('takePhoto({})'.format(quality))

    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

model1 = load_model("/content/models/model1.h5")

def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224,224))
    img = img / 255.0
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    pred = model1.predict(img)
    return np.argmax(pred)

In [ ]:
while True:
    path = take_photo()
    result = predict_image(path)
    if result == 0:
        print("Prediction: With Mask")
    else:
        print("Prediction: Without Mask")
    print("Prediction:", result)

<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
Prediction: With Mask
Prediction: 0


<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Prediction: With Mask
Prediction: 0


<IPython.core.display.Javascript object>

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Prediction: With Mask
Prediction: 0


<IPython.core.display.Javascript object>

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction: With Mask
Prediction: 0


<IPython.core.display.Javascript object>

MessageError: NotAllowedError: Permission dismissed